# M19b - Validation-safe-region controls

**Author:** Ildefons Magrans de Abril  
**Affiliation:** Universitat Politècnica de Catalunya - BarcelonaTech (UPC)

**Submission evidence.** Same-width interval and inside/outside movement controls are regenerated from the current executable protocol.

In [1]:
from pathlib import Path
import sys, numpy as np, pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import tcr_core as tcr
REPRO=ROOT/'results'/'reproduced'; REPRO.mkdir(parents=True,exist_ok=True)
def bootstrap_mean(x,n_boot=30000,seed=1):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    rng=np.random.default_rng(seed); idx=rng.integers(0,len(x),size=(n_boot,len(x)))
    b=x[idx].mean(axis=1)
    return float(x.mean()),float(np.quantile(b,.025)),float(np.quantile(b,.975))

In [2]:
SEED=20260622; TRIALS=12
TASKS=['controlled_d20_white_plus_distractor','memory_d10','narma10','lorenz_x']
CONFIG=dict(N=60,K=13,lengths=(1200,500,500),washout=100,input_scale=.8,ridge=1e-5)
rng=np.random.default_rng(SEED+991); rows=[]
for task in TASKS:
    for trial in range(TRIALS):
        case=tcr.evaluate_case(task,trial,'temperature',SEED,return_predictions=True,**CONFIG)
        scores=np.asarray(case['test_scores']); safe=np.asarray(case['safe_idxs']); d=int(case['default_idx'])
        outside=np.setdiff1d(np.arange(len(scores)),safe); inside=safe[safe!=d]; inside=safe if len(inside)==0 else inside
        ii=int(rng.choice(inside)); oi=int(rng.choice(outside)) if len(outside) else d
        rows.append({'task':task,'trial':trial,'near_contained':case['near_contained'],'exact_contained':case['exact_contained'],'matched_near_rate':case['matched_near_rate'],'matched_oracle_rate':case['matched_oracle_rate'],'safe_minus_matched_gain':case['safe_minus_matched_gain'],'random_inside_gain':float(scores[ii]-scores[d]),'random_outside_gain':float(scores[oi]-scores[d])})
rep=pd.DataFrame(rows); rep['inside_minus_outside_gain']=rep.random_inside_gain-rep.random_outside_gain
rep.to_csv(REPRO/'m19b_replication_case_metrics.csv',index=False)
ci=bootstrap_mean(rep.safe_minus_matched_gain,seed=1902); ci2=bootstrap_mean(rep.inside_minus_outside_gain,seed=1903)
summary=pd.DataFrame([{'n_cases':len(rep),'true_near_containment':rep.near_contained.mean(),'matched_near_containment':rep.matched_near_rate.mean(),'true_exact_containment':rep.exact_contained.mean(),'matched_exact_containment':rep.matched_oracle_rate.mean(),'mean_safe_minus_matched_gain':ci[0],'safe_minus_matched_ci_low':ci[1],'safe_minus_matched_ci_high':ci[2],'mean_inside_minus_outside_gain':ci2[0],'inside_minus_outside_ci_low':ci2[1],'inside_minus_outside_ci_high':ci2[2]}])
summary.to_csv(REPRO/'m19b_replication_summary.csv',index=False)
display(summary.round(6))

,n_cases,true_near_containment,matched_near_containment,true_exact_containment,matched_exact_containment,mean_safe_minus_matched_gain,safe_minus_matched_ci_low,safe_minus_matched_ci_high,mean_inside_minus_outside_gain,inside_minus_outside_ci_low,inside_minus_outside_ci_high
0,48,0.770833,0.404157,0.5625,0.180598,0.029665,0.020231,0.03993,0.036261,0.021413,0.051653
